# BloodMNIST: Random Forest baseline
Cleaned course-project workflow. Historical outputs are preserved separately in `archive/` and `results/reference/`.
Run all cells in order. No new training results are embedded in this notebook.


# Final Project

## Random Forest

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from medmnist import INFO
import medmnist
from torchvision import transforms

# Dataset
data_flag = 'bloodmnist'
info = INFO[data_flag]
DataClass = getattr(medmnist, info['python_class'])

# Transform: convert PIL image to tensor
transform = transforms.Compose([
    transforms.ToTensor()
])

# Load datasets
train_dataset = DataClass(split='train', transform=transform, download=True)
test_dataset = DataClass(split='test', transform=transform, download=True)

# Convert images to numpy and flatten
def dataset_to_numpy(dataset):
    images = []
    labels = []
    for i in range(len(dataset)):
        img, label = dataset[i]
        images.append(img.numpy().flatten()) 
        labels.append(label[0])  
    return np.stack(images), np.array(labels)

X_train, y_train = dataset_to_numpy(train_dataset)
X_test, y_test = dataset_to_numpy(test_dataset)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)


In [ ]:
# prints evaluation metrics

from sklearn.metrics import roc_auc_score, precision_score, recall_score
from sklearn.preprocessing import label_binarize

y_score = clf.predict_proba(X_test)  
y_pred = clf.predict(X_test)
n_classes = len(info['label'])

y_test_bin = label_binarize(y_test, classes=np.arange(n_classes))

# AUC
per_class_auc = roc_auc_score(y_test_bin, y_score, average=None, multi_class='ovr')
overall_auc = roc_auc_score(y_test_bin, y_score, average='macro', multi_class='ovr')


# overall precision/recall
precision_macro = precision_score(y_test, y_pred, average='macro')
recall_macro = recall_score(y_test, y_pred, average='macro')

# results
# per-class auc
print("Per-Class AUC:")
for i, auc in enumerate(per_class_auc):
    class_name = info['label'][str(i)]
    print(f"  Class {i}: AUC = {auc:.4f}")

# overall auc
print(f'Overall AUC: {overall_auc:.4f}')

# per class precision/recall
print(classification_report(y_test, y_pred, digits=4))

# overall precision/recall
print("\n Overall Precision & Recall:")
print(f"Precision: {precision_macro:.4f}")
print(f"Recall: {recall_macro:.4f}")


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report

out = Path("results/rerun")
out.mkdir(parents=True, exist_ok=True)
pd.DataFrame([{"model": "random_forest",
    "accuracy": accuracy_score(y_test, y_pred),
    "macro_f1": f1_score(y_test, y_pred, average="macro"),
    "macro_auc": overall_auc, "test_images": len(y_test)
}]).to_csv(out / "random_forest_metrics.csv", index=False)
pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).T.to_csv(
    out / "random_forest_class_metrics.csv")
print("New aggregate results saved to", out)


## Interpretation
Compare results with the archived tables, but expect differences across runs and package versions. The CNN seed was added during cleanup. Do not select hyperparameters using the test set; use the official validation split for further development. No external clinical validation is claimed.
